In [ ]:
from pathlib import Path
import csv
import sys

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.warp import reproject, Resampling
from affine import Affine
from model import FireForecast, MemoryAdapter

# WSTS dependency, if needed: %pip install segmentation-models-pytorch==0.3.3


In [ ]:
data_dir = Path("data")
baseline_dir = data_dir / "baselines"
sys.path.insert(0, str(baseline_dir.resolve()))
input_dir = data_dir / "baseline_inputs"
checkpoint = Path("best.pt")
device = torch.device("cuda")
threshold = 0.5
history_days = None  # All available history. Set an integer only for a comparison.
score_after_days = 1  # Use 5 for matching the pretrained baselines' forecast dates.
models_to_run = ["persistence", "ours", "wsts", "dailywildfirecan"]

# Same held-out fires as train.ipynb; these are its validation fires, not a new test set.
fires = sorted(p for p in data_dir.iterdir() if len(list(p.glob("*.npz"))) > 1)
np.random.default_rng(42).shuffle(fires)
fires = fires[int(0.8 * len(fires)):]
# To evaluate specific fires instead: fires = [data_dir / "FIRE_ID", ...]


In [ ]:
def load_day(path):
    with np.load(path) as data:
        return {key: data[key] for key in ("image", "date", "active", "burned", "new")}


def fire_windows(folder):
    files = sorted(folder.glob("*.npz"))
    days = [load_day(files[0])]
    for path in files[1:]:
        history = days if history_days is None else days[-history_days:]
        target = load_day(path)
        yield (path.stem, np.stack([d["image"] for d in history]),
               np.array([d["date"].item() for d in history]), target)
        days.append(target)


In [ ]:
# Inference-only version of the authors' SMPTempModel: same parameters and forward pass.
# Sources: github.com/slahrichi/WildfireSpreadTS, MIT; pinned source is cached under data/.
class WSTSForecast(nn.Module):
    def __init__(self, LTAE2d, TemporalAggregator):
        super().__init__()
        import segmentation_models_pytorch as smp
        self.model = smp.Unet(encoder_name="resnet18", encoder_weights=None,
                              in_channels=40, classes=1)
        self.ltae = LTAE2d(in_channels=512, n_head=16, d_k=4, mlp=[256, 512],
                           dropout=0.2, d_model=256, T=1000,
                           return_att=True, positional_encoding=True)
        self.temporal_aggregator = TemporalAggregator(mode="att_group")

    def forward(self, x, positions=None):
        stages = [self.model.encoder(x[:, t]) for t in range(x.shape[1])]
        positions = torch.arange(x.shape[1], device=x.device)[None].expand(x.shape[0], -1)
        last, attention = self.ltae(torch.stack([s[-1] for s in stages], dim=1),
                                    batch_positions=positions)
        skips = [self.temporal_aggregator(torch.stack([s[i] for s in stages], dim=1),
                                          attn_mask=attention)
                 for i in range(1, len(stages[0]) - 1)]
        decoded = self.model.decoder(stages[0][0], *skips, last)
        return self.model.segmentation_head(decoded)


def load_model(name):
    if name == "persistence":
        return None
    if name == "ours":
        saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
        state = saved.get("model", saved)
        memory = MemoryAdapter(high=1, low=-1, dims=3, size=4096, device=device,
                               shape={"encoder": 512}, steps=8)
        net = FireForecast(input_channels=58, memory=memory)
    elif name == "wsts":
        from wsts_backbones.ltae import LTAE2d
        from wsts_backbones.utae import Temporal_Aggregator
        net = WSTSForecast(LTAE2d, Temporal_Aggregator)
        state = torch.load(baseline_dir / "wsts_fold1.pt", map_location="cpu", weights_only=True)
    elif name == "dailywildfirecan":
        from can_utae import UTAE
        net = UTAE(input_dim=50, num_classes=1, encoder_widths=[64, 128, 256, 512])
        state = torch.load(baseline_dir / "can_utae_run1.pt", map_location="cpu", weights_only=True)["model_state_dict"]
    net.load_state_dict(state)
    return net.to(device).eval()


In [ ]:
# Prepared baseline_inputs/<model>/<fire>/<TARGET-DATE>.npz:
# x = normalized [5,40,H,W] (WSTS) or [3,50,H,W] (CAN), positions = [T],
# crs = CRS string, transform = six affine coefficients. Use historical inputs only.
def pretrained_prediction(name, net, fire, date, shape):
    with np.load(input_dir / name / fire / f"{date}.npz") as data:
        x = data["x"].astype(np.float32)
        positions = torch.as_tensor(data["positions"], dtype=torch.float32, device=device)[None]
        source_crs = str(data["crs"].item())
        source_transform = Affine(*data["transform"])

    # Cover the whole image with tiles; retain all pixels, including the edges.
    height, width = x.shape[-2:]
    probability = np.empty((height, width), dtype=np.float32)
    for top in range(0, height, 256):
        for left in range(0, width, 256):
            tile = torch.from_numpy(x[..., top:top+256, left:left+256]).to(device)[None]
            h, w = tile.shape[-2:]
            tile = F.pad(tile, (0, 256-w, 0, 256-h))
            probability[top:top+h, left:left+w] = net(tile, positions).sigmoid()[0, 0, :h, :w].cpu().numpy()

    reference = next((data_dir / "raw" / "ts-satfire" / fire / "VIIRS_Day").glob("*.tif"))
    with rasterio.open(reference) as grid:
        aligned = np.full(shape, np.nan, dtype=np.float32)
        reproject(probability, aligned, src_transform=source_transform, src_crs=source_crs,
                  dst_transform=grid.transform, dst_crs=grid.crs,
                  dst_nodata=np.nan, resampling=Resampling.bilinear)
    return aligned[None]


In [ ]:
def metrics(predictions, targets):
    p = np.concatenate([a.ravel() for a in predictions])
    y = np.concatenate([a.ravel() for a in targets]).astype(bool)
    binary = p >= threshold
    tp = np.count_nonzero(binary & y)
    fp = np.count_nonzero(binary & ~y)
    fn = np.count_nonzero(~binary & y)
    divide = lambda a, b: float(a / b) if b else np.nan

    # Exact AP, grouping tied scores (including persistence's binary scores).
    order = np.argsort(-p, kind="stable")
    ends = np.r_[np.flatnonzero(np.diff(p[order])), len(p)-1]
    hits = np.cumsum(y[order], dtype=np.int64)[ends]
    ap = divide(np.sum(np.diff(np.r_[0, hits]) * hits / (ends+1)), y.sum())
    return dict(AP=ap, F1=divide(2*tp, 2*tp+fp+fn), IoU=divide(tp, tp+fp+fn),
                precision=divide(tp, tp+fp), recall=divide(tp, tp+fn))


In [ ]:
results = []
with torch.inference_mode():
    for name in models_to_run:
        if name in ("wsts", "dailywildfirecan") and not (input_dir / name).exists():
            print(f"Skipping {name}: prepared inputs are missing.")
            continue
        net = load_model(name)
        for folder in fires:
            print(name, folder.name, flush=True)
            predictions, targets = {}, {}
            for day, (date, history, dates, target) in enumerate(fire_windows(folder), start=1):
                if name == "wsts" and day < max(5, score_after_days):
                    continue
                if name == "dailywildfirecan" and day < max(3, score_after_days):
                    continue
                previous = history[-1, 28:29]
                if name == "persistence":
                    output = dict(active=history[-1, 27:28], burned=previous,
                                  new=np.zeros_like(previous))
                elif name == "ours":
                    x = torch.from_numpy(history).to(device)[None]
                    valid = torch.isfinite(x)
                    x = torch.cat([torch.where(valid, x, 0), valid.float()], dim=2)
                    positions = torch.as_tensor(dates-dates[-1], dtype=torch.float32, device=device)[None]
                    values = net(x, positions)
                    output = {key: value[0].cpu().numpy()
                              for key, value in zip(("active", "burned", "new"), values)}
                    del x, valid, positions, values
                else:
                    probability = pretrained_prediction(name, net, folder.name, date, previous.shape[-2:])
                    if name == "wsts":
                        output = dict(active=probability)
                    else:
                        new = (1-previous) * probability
                        output = dict(new=new, burned=previous+new)
                # Still run earlier forecasts to warm our memory, even when scoring later dates.
                if day < score_after_days:
                    continue
                for key, probability in output.items():
                    predictions.setdefault(key, []).append(probability)
                    targets.setdefault(key, []).append(target[key])
            if name == "ours":
                memory = net.memory.memory
                memory.voltage = torch.zeros_like(memory.voltage)
                memory.spikes = torch.zeros_like(memory.spikes)
                memory.y = {key: memory.voltage.new_zeros(width) for key, width in memory.shape.items()}
                del memory
            for key in predictions:
                results.append(dict(model=name, fire=folder.name, output=key, windows=len(predictions[key]),
                                    **metrics(predictions[key], targets[key])))
        del net
        torch.cuda.empty_cache()


In [ ]:
# Average each metric across fires; AP here is per-fire macro AP, not pooled AP.
for name, output in dict.fromkeys((r["model"], r["output"]) for r in results):
    group = [r for r in results if r["model"] == name and r["output"] == output]
    averages = {}
    for metric in ("AP", "F1", "IoU", "precision", "recall"):
        values = [r[metric] for r in group if np.isfinite(r[metric])]
        averages[metric] = round(float(np.mean(values)), 4) if values else np.nan
    print(name, output, sum(r["windows"] for r in group), "forecasts", averages)

with (data_dir / "test_results.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=["model", "fire", "output", "windows", "AP", "F1", "IoU", "precision", "recall"])
    writer.writeheader()
    writer.writerows(results)
